# Tree-Based Modeling

## Why Not Linear/Logistic Regression?

This algorithm is used when the relationship between the features and outcome is non-linear. In the practical sense, decision tree is never really used, but is a starting point to more complex trees: Random Forest, XGBoost, Lightgbm

## Decision Tree

### Regression

#### Objective

Regression decision tree predicts mean numerical value for an outcome (e.g., weight) for each region by identifying splits in the feature space (e.g. height).

#### Algorithm

- Building the Tree
  - objective
    - The purpose of this initial tree is to create as many leaves as possible that minimizes RSS: $\sum_{j=1}^{J}\sum_{i\in R_j}(y_i-\hat{y}_{R_j})^2$
      - $\hat{y}_{R_j}$ is the mean response for the training observations within the jth region
  - process
    - ...in English:
      - Method considers all predictors: $X_1,...,X_p$ and all possible values for the cutpoint `s` such that the resulting two new nodes' RSS's combined is low
    - ...in Mathematics:
      - For any $j$ and $s$, we define the two new nodes as...
        - $R_1(j,s)= \{X|X_j<s\}$ and $R_2(j,s)= \{X|X_j \geq s\}$
      - We seek to find $j$ and $s$ that minimizes...
          - $\sum_{i:x_i\in R_1(j,s)}(y_i-\hat{y}_{R_1})^2 + \sum_{i:x_i\in R_2(j,s)}(y_i-\hat{y}_{R_2})^2$
      - Repeat this method within the resulting regions.
- Pruning
  - objective
    - The issue with building the initial tree is that the resulting tree is going overfit, so `pruning` is introduced to create a simpler tree that is generalizable during the building process.
  - process
    - there are two types of pruning
      - Structural Pruning
        - main idea
          - this type of pruning is controlled by the user as they are absolute rules based on input parameters
        - popular rules
          - `max_depth`
            - no more splits below this level (e.g., 5 levels of nodes)
          - `min_samples_leaf`
            - no leaf can have less than this # of samples to ensure tiny leaves are not made
          - `min_samples_split`
            - no split can occur if # of samples is less than this minimum
      - Gain-Based Pruning
        - main idea
          - A split must reduce loss enough to justify increasing model complexity
        - process
          - Gain $ = \sum_{i:x_i\in R_0}(y_i-\hat{y}_{R_0})^2 - (\sum_{i:x_i\in R_1(j,s)}(y_i-\hat{y}_{R_1})^2 + \sum_{i:x_i\in R_2(j,s)}(y_i-\hat{y}_{R_2})^2)$
          - Gain = $MSE_{parent} - (MSE_{child_1} + MSE_{child_2})$
          - Gain ≤ 0 → no split

### Classification

#### Objective

Classification decision tree predicts a category/class label for an outcome (e.g., spam vs not spam) for each region by identifying splits in the feature space.

- In a given region (leaf) $R_j$, we estimate class probabilities:
  - $\hat{p}_{k, R_j} = \frac{1}{|R_j|}\sum_{i \in R_j} \mathbf{1}(y_i = k)$
    - $k$ is the class of interest (e.g., spam)
- The predicted class for that region is typically:
  - $\hat{y}_{R_j} = \arg\max_k \hat{p}_{k, R_j}$
    - for the $R_j$ region, choose the class with the highest probability

#### Algorithm

- Building the Tree
  - objective
    - Instead of minimizing RSS (regression), the purpose of the classification tree is to create leaves that are as "pure" as possible (i.e., dominated by a single class).
    - This is done by minimizing a classification impurity measure within each node.
  - process
    - ...impurity measures (for a node/region $R$)
      - Gini Index
        - $G(R) = \sum_{k=1}^{K} \hat{p}_{k,R}(1-\hat{p}_{k,R})$
          - $\hat{p}_{k,R}$ = represents the proportion of training observations in the $R$th region that are from the $k$th class
          - measures the purity of the node, the smaller the gini index ($\hat{p}_{k,R} \in {0,1}$) the better
    - ...in English:
      - Method considers all predictors: $X_1,...,X_p$ and candidate cutpoints `s` such that the resulting two new nodes’ combined impurity is low.
    - ...in Mathematics:
      - For any $j$ and $s$, we define the two new nodes as...
        - $R_1(j,s)= \{X|X_j<s\}$ and $R_2(j,s)= \{X|X_j \geq s\}$
      - We seek to find $j$ and $s$ that minimizes the weighted impurity:
        - $\frac{N_1}{N}I(R_1(j,s)) + \frac{N_2}{N}I(R_2(j,s))$
          - where $I(\cdot)$ is Gini or Cross-Entropy, $N$ is # samples in the parent, and $N_1, N_2$ are # samples in each child.
      - Repeat this method within the resulting regions.

- Pruning
  - objective
    - The issue with building the initial tree is that the resulting tree can overfit, so pruning is introduced to create a simpler tree that generalizes better.
  - process
    - Structural Pruning
      - main idea
        - this type of pruning is controlled by the user as absolute rules based on input parameters
      - popular rules
        - `max_depth`
          - no more splits below this level (e.g., 5 levels of nodes)
        - `min_samples_leaf`
          - no leaf can have less than this # of samples to ensure tiny leaves are not made
        - `min_samples_split`
          - no split can occur if # of samples is less than this minimum
    - Gain-Based Pruning
      - main idea
        - A split must reduce impurity enough to justify increasing model complexity
      - process
        - Gain (impurity reduction) = $I(R_0) - (\frac{N_1}{N}I(R_1) + \frac{N_2}{N}I(R_2))$
        - Gain ≤ 0 → no split
    - Cost-Complexity (Weakest Link) Pruning (common in CART)
      - main idea
        - choose a subtree that trades off fit vs complexity
      - process
        - Define a cost:
          - $\sum_{m\in \text{leaves}(T)} N_m\cdot I(R_m) + \alpha |T|$
            - $|T|$ is number of leaves
            - $\alpha$ controls how much we penalize complexity
        - Use cross-validation to choose $\alpha$

## Bagging

### Objective

- A decision tree is prone to high variance.
- This means that consistency of predictions across different datasets is low, so to address this is to create a process with low variance.
- `Bagging` is a technique where we create multiple independent models (each trained on a subset of the training data) then take the majority vote (classification) or average (regression) as final prediction. 
- One type of ensemble method

## Random Forest

### Objective

- A single decision tree is prone to high variance.
- `Bagging` reduces variance by averaging many trees trained on bootstrap samples, but the trees can still be highly correlated (they often pick the same strong predictors early).
- `Random Forest` extends bagging by adding **feature randomness** to decorrelate the trees, which **further reduces variance and improves generalization.**

### Algorithm

- Setup
  - given training data $\{(x_i,y_i)\}_{i=1}^{n}$ with $p$ predictors
  - choose # of trees $B$ and # of candidate features per split $m$ (often called `max_features` or `mtry`)

- Building the Forest (train $B$ trees)
  - for $b = 1,...,B$:
    - Bootstrap the data
      - sample $n$ observations **with replacement** from the training set to get bootstrap dataset $\mathcal{D}^{(b)}$
    - Grow a decision tree on $\mathcal{D}^{(b)}$
      - at each split in the tree:
        - randomly select $m$ predictors from the full set of $p$ predictors
        - find the best split using only those $m$ predictors
          - regression: choose split that minimizes RSS (same as decision tree regression)
          - classification: choose split that minimizes impurity (Gini / cross-entropy)
      - grow the tree deep (often no pruning; rely on averaging to control variance)

- Prediction (aggregate across trees)
  - Regression
    - average the $B$ tree predictions:
      - $\hat{f}(x) = \frac{1}{B}\sum_{b=1}^{B}\hat{f}^{(b)}(x)$
  - Classification
    - majority vote across the $B$ trees:
      - $\hat{y}(x) = \text{mode}\{\hat{y}^{(1)}(x),...\,,\hat{y}^{(B)}(x)\}$
    - if you want probabilities
      - average class probabilities from each tree and then pick the largest probability

### Notes / Intuition

- Why the random feature subset matters
  - bagging reduces variance by averaging
  - random forest reduces variance further by making trees **less correlated** (different trees see different candidate predictors at each split)

- Out-of-Bag (OOB) Error
  - each bootstrap sample leaves out about ~1/3 of the training points
  - for each training point, predict using only trees where that point was **not** included in the bootstrap sample
  - average these errors to get an OOB estimate of test error (useful for quick evaluation and tuning)

### Hyperparameters (common)

- `n_estimators` / $B$
  - more trees → usually better and more stable, but slower
- `max_features` / $m$
  - smaller $m$ → more randomness (less correlation), but can increase bias if too small
- Tree controls
  - `max_depth`, `min_samples_split`, `min_samples_leaf` (controls overfitting + speed)
- `bootstrap`
  - whether to use bootstrap sampling (classic RF uses bootstrap)

## Boosting

### Objective

- While bagging / random forests build many trees **independently** to reduce variance, `Boosting` builds trees **sequentially** to reduce bias (and often variance).
- Main idea: combine many **weak learners** (often shallow trees) into a strong model by repeatedly adding a new model that corrects the errors of the current ensemble.
- Rule of thumb: `bagging/RF` for stability + low tuning, `boosting` for maximum accuracy when you can tune and control overfitting.

## Gradient Boosting

### Objective

- `Gradient Boosting` is the most common boosting framework used for decision trees.
- Main idea: build an additive model by adding one small tree at a time to reduce a chosen loss function.
- Instead of "fit the residuals" being a special trick for squared error, gradient boosting generalizes this idea to **any differentiable loss** by fitting to the **negative gradient** of the loss.

### Algorithm (high-level)

- Setup
  - choose a loss function $L(y, \hat{f}(x))$
    - regression examples
      - squared error: $(y-\hat{f}(x))^2$
      - absolute error: $|y-\hat{f}(x)|$ (not differentiable everywhere; handled with variants)
      - Huber loss (robust to outliers)
  - choose number of trees $B$, learning rate $\eta$, and tree complexity (depth/leaves)

- Initialize
  - start with a simple constant model
    - $\hat{f}_0(x) = \arg\min_c \sum_{i=1}^{n} L(y_i, c)$
      - for squared error, this is the mean of $y$

- For $b = 1,...,B$
  - compute "pseudo-residuals" (what the model still needs to fix)
    - $r_{i}^{(b)} = -\left[\frac{\partial L(y_i, \hat{f}(x_i))}{\partial \hat{f}(x_i)}\right]_{\hat{f}=\hat{f}_{b-1}}$
    - intuition
      - $r_i^{(b)}$ tells us how each prediction $\hat{f}_{b-1}(x_i)$ should move to reduce loss
      - if $r_i^{(b)} > 0$ → increasing $\hat{f}(x_i)$ reduces loss; if $r_i^{(b)} < 0$ → decreasing $\hat{f}(x_i)$ reduces loss
      - this is why gradient boosting is like "gradient descent in function space"
    - example (squared error)
      - if $L(y,\hat{f}) = \frac{1}{2}(y-\hat{f})^2$, then $r_i^{(b)} = y_i - \hat{f}_{b-1}(x_i)$
        - so for squared error, pseudo-residuals are just the usual residuals
        - numerical example:
          - predicted = 100, observed = 80
          - dL = predicted - observed = 100 - 80 = 20
          - $r_i^{(b)}$ = -dL = -20
          - so for the next tree, we're trying to lower the overall prediction
  - fit a small regression tree $h_b(x)$ to predict $r_{i}^{(b)}$ from $x_i$
    - (tree tries to approximate the direction that most reduces loss)
  - update the model
    - $\hat{f}_b(x) = \hat{f}_{b-1}(x) + \eta\, h_b(x)$

### Notes / Intuition

- Gradient boosting is "gradient descent in function space".
- Shallow trees (stumps / small depth) are common because each tree is a small corrective step.
- Smaller $\eta$ usually improves generalization but requires a larger $B$.

### Hyperparameters (common)

- `n_estimators` / $B$
  - number of boosting steps
- `learning_rate` / $\eta$
  - shrinkage/step size
- Tree complexity
  - `max_depth` or `max_leaf_nodes`, `min_samples_leaf`
- Regularization via randomness (often)
  - `subsample` (row sampling) and sometimes `max_features` (column sampling)


## XGBoost

### Objective (intuition first)

- `XGBoost` builds a strong model by adding **many small decision trees** one-by-one.
- Each new tree is a **correction** to the current predictions.
- The key advantages vs "plain" gradient boosting are:
  - explicit regularization (to reduce overfitting)
  - better split scoring (uses both gradient + curvature)
  - strong engineering (fast, handles missing/sparse values well)

### Big picture

- XGBoost is still an additive model:
  - $\hat{y}_i = \hat{f}(x_i) = \sum_{b=1}^{B} f_b(x_i)$
- Think of it as:
  - baseline prediction + correction tree + correction tree + ...

### What a new tree is trying to learn

- At boosting step $b$, you already have predictions $\hat{y}_i^{(b-1)}$.
- You now ask: "For each training point, should my prediction go **up** or **down**, and by how much?"

- For squared error regression, this is the familiar residual idea:
  - residual $= y_i - \hat{y}_i^{(b-1)}$
  - if residual is positive → next tree should push prediction up
  - if residual is negative → next tree should push prediction down

- For general losses, XGBoost uses the same idea but with derivatives:
  - $g_i$ (gradient) = the direction to change $\hat{y}_i$ to reduce loss
  - $h_i$ (hessian) = how "sensitive" the loss is around $\hat{y}_i$ (stability / confidence)

### Objective Function (why regularization matters)

- XGBoost optimizes training loss + a penalty for complexity:
  - $\mathcal{L} = \sum_{i=1}^{n} L(y_i, \hat{y}_i) + \sum_{b=1}^{B} \Omega(f_b)$

- A common regularizer for a tree $f$ is:
  - $\Omega(f) = \gamma T + \frac{1}{2}\lambda\sum_{j=1}^{T} w_j^2$ (and sometimes $+\alpha\sum_{j=1}^{T}|w_j|$)
    - $T$ = # leaves (more leaves = more complex)
    - $w_j$ = value predicted by leaf $j$
    - $\gamma,\lambda,\alpha$ penalize complexity

### Algorithm (how one tree is added) — more intuition

At boosting step $b$, you already have predictions $\hat{y}_i^{(b-1)}$. You want to add one new tree that makes the loss smaller.

1. Compute how each point wants its prediction to change
    - Compute derivatives of the loss at the current predictions
      - $g_i = \frac{\partial L(y_i, \hat{y}_i)}{\partial \hat{y}_i}\Big|_{\hat{y}_i=\hat{y}_i^{(b-1)}}$ (gradient)
        - If $g_i > 0$, decreasing $\hat{y}_i$ reduces loss; if $g_i < 0$, increasing $\hat{y}_i$ reduces loss
      - $h_i = \frac{\partial^2 L(y_i, \hat{y}_i)}{\partial \hat{y}_i^2}\Big|_{\hat{y}_i=\hat{y}_i^{(b-1)}}$ (hessian / curvature)
        - "How sensitive" the loss is around the current prediction (bigger $h_i$ → loss changes faster)
    - Quick example (squared error)
      - If $L(y,\hat{y}) = \frac{1}{2}(y-\hat{y})^2$, then $g_i = \hat{y}_i - y_i$ and $h_i = 1$
      - If $y=80$ and $\hat{y}=100$, then $g = 20$ (positive) → the next tree should push the prediction down

2. Grow a tree that groups similar corrections
    - A tree partitions points into leaves (regions) $R_1, R_2, ...$
    - Inside a leaf $j$, XGBoost applies the *same* correction value $w_j$ to every point in that leaf

3. For each leaf, compute the best correction value
    - Aggregate the derivatives inside a leaf $j$
      - $G_j = \sum_{i\in R_j} g_i$ (total "push" direction)
      - $H_j = \sum_{i\in R_j} h_i$ (total curvature)
    - The best leaf value is
      - $w_j^* = -\frac{G_j}{H_j+\lambda}$
        - Minus sign = move opposite the gradient (reduce loss)
        - $\lambda$ shrinks leaf values (regularization)

4. Decide splits using Gain (is the split worth it?)
    - When you split a parent node $P$ into left $L$ and right $R$, you compare
      - one correction for everyone in $P$ vs separate corrections for $L$ and $R$
    - Split gain (worth-it score)
      - $\text{Gain} = \frac{1}{2}\left(\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{G_P^2}{H_P+\lambda}\right) - \gamma$
      - Gain ≤ 0 → no split
      - $\gamma$ penalizes adding a split (prevents unnecessary splits)

5. Update predictions with shrinkage (small step)
    - After you build tree $f_b$, update predictions
      - $\hat{y}_i^{(b)} = \hat{y}_i^{(b-1)} + \eta\, f_b(x_i)$
    - $\eta$ (learning rate) controls how much we trust each new tree (smaller $\eta$ → safer updates, usually needs more trees)

### Regression vs Classification

- Regression
  - choose a regression loss (common: squared error)
  - final prediction is the sum of tree outputs
- Classification
  - commonly uses logistic loss
  - model outputs a score (logit); probability is:
    - $p(y=1|x) = \sigma(\hat{f}(x))$

### Regularization / Hyperparameters (common)

- `n_estimators` ($B$)
  - number of boosting rounds
- `learning_rate` ($\eta$)
  - size of each correction step (smaller usually needs more trees)

- Tree complexity
  - `max_depth` (how complex each correction tree can be)
  - `min_child_weight` (prevents tiny/noisy splits)
  - `gamma` (min split gain; discourages unnecessary splits)

- Randomness (helps generalization)
  - `subsample` (row sampling)
  - `colsample_bytree`, `colsample_bynode` (column sampling)

- Leaf penalties
  - `reg_lambda` ($\lambda$) L2
  - `reg_alpha` ($\alpha$) L1

### Evaluation

- Use validation / CV to tune hyperparameters.
- Best practice
  - `early_stopping_rounds` to stop when validation metric stops improving
- Metrics
  - regression: RMSE / MAE
  - classification: logloss, ROC-AUC / PR-AUC (depending on class imbalance)
